In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

from src.arranger.arranger import arrange_chord_sequence
from src.synth.midi_renderer import *
from src.synth.midi_renderer import synthesize_with_drums
from src.audio.pitch import *
from src.audio.chroma import extract_chroma
from src.harmony.viterbi import suggest_chords_viterbi
from IPython.display import Audio
import librosa
import pretty_midi

## Teste do arranjador

In [ ]:
audio_path = '../data/audio-segment1-acima-do-sol-lead.mp3'
TOM='G'
BPM=92
OFFSET=1
UPC = 2
y, sr = librosa.load(audio_path)
listen_beats(y, sr, BPM, offset=OFFSET)
time, frequency, confidence = extract_f0_pyin(y, sr)
df_compassos = extract_chroma(
    time,
    frequency,
    confidence,
    BPM,
    tom=TOM,
    offset=OFFSET,
    unidades_por_compasso=UPC,
    quantizar_para_tom=True
)
print(df_compassos)

acordes, caminho, funcoes, seg = suggest_chords_viterbi(
  df_compassos, 
  tom_raiz='G', 
  tamanho_bloco=1,
  custo_troca_fora_tempo_forte=0.6, 
  permitir_diminutos=False,
)

df_compassos['acorde'] = acordes
df_compassos['funcao'] = df_compassos['acorde'].map(funcoes)

print(df_compassos[['compasso', 'nota_dominante', 'acorde', 'funcao']])


    compasso nota_dominante acorde funcao
0          1              A     Am     SD
1          1              G     Em      T
2          2              B      G      T
3          2              A     Am     SD
4          3              D      D      D
5          3              D      D      D
6          4              G      G      T
7          4              B      G      T
8          5              A     Am     SD
9          5              A      D      D
10         6              A     Em      T
11         6              A     Am     SD
12         7              G      G      T
13         7             F#      G      T


In [46]:
atividade_vocal = extract_vocal_activity(
    time,
    frequency,
    confidence,
    BPM,
    OFFSET,
    unidades_por_compasso=UPC,
    min_voiced_ratio=0.7
)

print(atividade_vocal)

[np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_]


In [47]:
acordes_sugeridos = acordes if UPC == 1 else seg
eventos_midi = arrange_chord_sequence(
    acordes_sugeridos=acordes_sugeridos,
    tom_raiz=TOM,
    atividade_vocal=atividade_vocal,
    bpm=BPM,
    unidades_por_compasso_vocal=UPC,
    offset=OFFSET
)

pm = export_events_to_midi(eventos_midi, '../data/midis/output.mid')
audio = synthesize_with_drums(pm, fs=22050, drum_gain=256.0)
## audio = pm.synthesize(fs=22050)
offset = int(0 * (60 / BPM) * 22050)

# 2. Aplica o pad no áudio sintetizado
audio_final = np.pad(audio, (offset, 0), mode="constant")

# 3. Igualar os tamanhos de audio_final e y para permitir a soma sem erro de broadcast
tamanho_max = max(len(audio_final), len(y))
audio_final_aligned = np.pad(audio_final, (0, tamanho_max - len(audio_final)))
y_aligned = np.pad(y, (0, tamanho_max - len(y)))

# 4. Mixagem final e reprodução
audio_mix = audio_final_aligned + 256.0 * y_aligned
Audio(audio_mix, rate=22050)

In [ ]:
pm = export_events_to_midi(eventos_midi, '../data/midis/output.mid')

# instrumento de bateria deveria vir zerado pelo sintetizador padrão
idx_bateria = next(i for i, inst in enumerate(pm.instruments) if inst.is_drum)
print("synthesize padrão (deve ser tudo zero):", 
      np.abs(pm.instruments[idx_bateria].synthesize(fs=22050)).sum())

audio = synthesize_with_drums(pm, fs=22050)
print("synthesize_with_drums (deve ter energia > 0):", np.abs(audio).sum())

synthesize padrão (deve ser tudo zero): 0.0
synthesize_with_drums (deve ter energia > 0): 15003981.973864546
